# Teleoperation

In this example we'll control the Jetbot remotely with a gamepad controller connected to our web browser machine.

### Create gamepad controller

The first thing we want to do is create an instance of the ``Controller`` widget, which we'll use to drive our robot.
The ``Controller`` widget takes a ``index`` parameter, which specifies the number of the controller.  This is useful in case you
have multiple controllers attached, or some gamepads *appear* as multiple controllers.  To determine the index
of the controller you're using,

1. Visit [http://html5gamepad.com](http://html5gamepad.com).  
2. Press buttons on the gamepad you're using
3. Remember the ``index`` of the gamepad that is responding to the button presses

Next, we'll create and display our controller using that index.

In [1]:
import ipywidgets.widgets as widgets

controller = widgets.Controller(index=0)  # replace with index of your controller

display(controller)

Controller()

In [2]:
import traitlets

# class MockMotor(traitlets.HasTraits):
#     value = traitlets.Float()

# class MockRobot:
#     def __init__(self):
#         self.left_motor  = MockMotor()
#         self.right_motor = MockMotor()

# robot = MockRobot()


from jetbot import Robot, Camera

robot = Robot()

In [13]:
# import traitlets
# from IPython.display import display





# # Controls the top speed of both wheels simultaneously.
# # 0.1 = very slow (10% power), 1.0 = full speed (100% power).
# # Start around 0.3–0.5 to keep things safe.
# max_speed_slider = widgets.FloatSlider(
#     value=0.5,   # starting speed at 50%
#     min=0.1,     # never goes below 10% (avoids motors stalling)
#     max=1.0,     # hard cap at 100%
#     step=0.05,   # each slider tick changes speed by 5%
#     description='Max Speed:',
#     readout_format='.2f'
# )

# left_scale_slider = widgets.FloatSlider(
#     value=1.0,
#     min=0.8,    # up to 20% slower
#     max=1.2,    # up to 20% faster
#     step=0.01,
#     description='Left Scale:',
#     readout_format='.2f'
# )

# right_scale_slider = widgets.FloatSlider(
#     value=0.84,
#     min=0.8,
#     max=1.2,
#     step=0.01,
#     description='Right Scale:',
#     readout_format='.2f'
# )

# display(widgets.VBox([max_speed_slider, left_scale_slider, right_scale_slider]))




# left_power_label  = widgets.Label(value='Left:  0.00')
# right_power_label = widgets.Label(value='Right: 0.00')

# display(widgets.VBox([left_power_label, right_power_label]))  




# def left_transform(x):
#     power = (x * max_speed_slider.value * left_scale_slider.value)
#     left_power_label.value  = f'Left:  {power:+.2f}'
#     return power

# def right_transform(x):
#     power = (x * max_speed_slider.value * right_scale_slider.value)
#     right_power_label.value = f'Right: {power:+.2f}'
#     return power

# left_link  = traitlets.dlink((controller.buttons[6], 'value'), (robot.left_motor,  'value'), transform=left_transform) 
# right_link = traitlets.dlink((controller.buttons[7], 'value'), (robot.right_motor, 'value'), transform=right_transform) 

IndexError: tuple index out of range

Even if the index is correct, you may see the text ``Connect gamepad and press any button``.  That's because the gamepad hasn't
registered with this notebook yet.  Press a button and you should see the gamepad widget appear above.

### Connect gamepad controller to robot motors

Now, even though we've connected our gamepad, we haven't yet attached the controls to our robot!  The first, and most simple control
we want to attach is the motor control.  We'll connect that to the left and right vertical axes using the ``dlink`` function.  The
``dlink`` function, unlike the ``link`` function, allows us to attach a transform between the ``source`` and ``target``.  Because
the controller axes are flipped from what we think is intuitive for the motor control, we'll use a small *lambda* function to
negate the value.

> WARNING: This next cell will move the robot if you touch the gamepad controller axes!

In [7]:
import ipywidgets as widgets
import traitlets
from IPython.display import display


# Controls the top speed of both wheels simultaneously.
# 0.1 = very slow (10% power), 1.0 = full speed (100% power).
# Start around 0.3–0.5 to keep things safe.
max_speed_slider = widgets.FloatSlider(
    value=0.3,   # starting speed at 50%
    min=0.1,     # never goes below 10% (avoids motors stalling)
    max=1.0,     # hard cap at 100%
    step=0.1,   # each slider tick changes speed by 5%
    description='Max Speed:',
    readout_format='.2f'
)

# Multiplies left motor output by a constant scale factor to correct hardware drift.
# 1.0 = no correction, 0.95 = 5% slower, 1.05 = 5% faster.
left_scale_slider = widgets.FloatSlider(
    value=1.0,
    min=0.6,     # up to 20% slower
    max=1.4,     # up to 20% faster
    step=0.01,
    description='Left Scale:',
    readout_format='.2f'
)

# Same as left scale but for the right motor.
right_scale_slider = widgets.FloatSlider(
    value=0.98,
    min=0.6,     # up to 20% slower
    max=1.4,     # up to 20% faster
    step=0.01,
    description='Right Scale:',
    readout_format='.2f'
)

# How much axes[2] (left stick horizontal) can boost/reduce each wheel.
# 0.5 = up to 50% speed difference between wheels when turning fully.
turn_boost_slider = widgets.FloatSlider(
    value=0.05,
    min=0.0,
    max=1.0,
    step=0.05,
    description='Turn Boost:',
    readout_format='.2f'
)


TURN_POWER = 0.9


display(widgets.VBox([max_speed_slider, left_scale_slider, right_scale_slider, turn_boost_slider]))

left_power_label  = widgets.Label(value='Left:  0.00')
right_power_label = widgets.Label(value='Right: 0.00')
display(widgets.VBox([left_power_label, right_power_label]))


def update_motors(change):
    # buttons[6/7] are analog triggers: 0.0 (released) to 1.0 (fully pressed)
    # each trigger drives one wheel independently (tank style)
    left_trigger  = controller.buttons[7].value
    right_trigger = controller.buttons[6].value

    # axes[1]: left stick vertical — forward/backward applied equally to both wheels
    # negated because gamepad axis is flipped (push forward = negative value)
    throttle = -controller.axes[1].value

    # axes[2]: left stick horizontal — positive = right, negative = left
    # boosts one wheel and reduces the other to steer
    turn = controller.axes[2].value

    left_power  = (left_trigger * TURN_POWER  + throttle + turn * turn_boost_slider.value) * max_speed_slider.value * left_scale_slider.value
    right_power = (right_trigger * TURN_POWER + throttle - turn * turn_boost_slider.value) * max_speed_slider.value * right_scale_slider.value

    # clamp to [-1, 1] so motors never receive illegal values
    left_power  = max(-1.0, min(1.0, left_power))
    right_power = max(-1.0, min(1.0, right_power))

    robot.left_motor.value  = left_power
    robot.right_motor.value = right_power

    left_power_label.value  = f'Left:  {left_power:+.2f}'
    right_power_label.value = f'Right: {right_power:+.2f}'


# all four inputs trigger the same update function
controller.buttons[6].observe(update_motors, names='value')
controller.buttons[7].observe(update_motors, names='value')
controller.axes[1].observe(update_motors, names='value')
controller.axes[2].observe(update_motors, names='value')

### recording videos of track

In [8]:
import cv2
import csv
import time
import threading
import numpy as np
import ipywidgets as widgets
from IPython.display import display
from jetbot import bgr8_to_jpeg
import os

camera = Camera()

os.makedirs('recordings', exist_ok=True)

# --- Settings ---
vid_length_slider = widgets.FloatSlider(
    value=2.0, min=0.5, max=30.0, step=0.5,
    description='Clip length (min):', readout_format='.1f', style={'description_width': 'initial'}
)
display(vid_length_slider)

# --- State ---
recording       = False
video_writer    = None
csv_file        = None
csv_writer      = None
segment_start   = None
segment_timer   = None
current_left    = 0.0   # updated by update_motors
current_right   = 0.0
current_inputs  = {'throttle': 0.0, 'turn': 0.0, 'left_trigger': 0.0, 'right_trigger': 0.0}

# --- UI ---
record_button = widgets.ToggleButton(
    value=False, description='⏺ Start Recording',
    button_style='danger', layout=widgets.Layout(width='200px')
)
status_label = widgets.Label(value='Not recording.')
clip_label   = widgets.Label(value='')
display(widgets.VBox([record_button, status_label, clip_label]))


def make_session_name():
    return 'recordings/' + time.strftime('%Y%m%d_%H%M%S')


def start_segment(session_base, segment_index):
    """Open a new VideoWriter and CSV for this segment."""
    global video_writer, csv_file, csv_writer, segment_start, segment_timer

    h, w = camera.value.shape[:2]
    vid_path = f'{session_base}_seg{segment_index:03d}.mp4'
    csv_path = f'{session_base}_seg{segment_index:03d}.csv'

    video_writer = cv2.VideoWriter(
        vid_path,
        cv2.VideoWriter_fourcc(*'mp4v'),
        10,           # fps — JetBot camera runs ~10fps
        (w, h)
    )
    csv_file   = open(csv_path, 'w', newline='')
    csv_writer = csv.writer(csv_file)
    csv_writer.writerow(['timestamp', 'left_motor', 'right_motor',
                         'throttle', 'turn', 'left_trigger', 'right_trigger'])

    segment_start = time.time()
    clip_label.value = f'Clip: {os.path.basename(vid_path)}'

    # schedule auto-split after vid_length minutes
    segment_timer = threading.Timer(
        vid_length_slider.value * 60,
        lambda: auto_split(session_base, segment_index)
    )
    segment_timer.daemon = True
    segment_timer.start()


def stop_segment():
    """Flush and close the current segment."""
    global video_writer, csv_file, segment_timer

    if segment_timer:
        segment_timer.cancel()
        segment_timer = None
    if video_writer:
        video_writer.release()
        video_writer = None
    if csv_file:
        csv_file.close()
        csv_file = None


def auto_split(session_base, segment_index):
    """Called by timer — save current segment and start a new one."""
    if not recording:
        return
    stop_segment()
    status_label.value = f'Auto-saved segment {segment_index}. Starting next...'
    start_segment(session_base, segment_index + 1)


def on_record_toggle(change):
    global recording, _session_base, _segment_index

    if change['new']:  # start recording
        recording        = True
        _session_base    = make_session_name()
        _segment_index   = 0
        record_button.description = '⏹ Stop Recording'
        status_label.value = 'Recording...'
        start_segment(_session_base, _segment_index)
    else:              # stop recording
        recording = False
        stop_segment()
        record_button.description = '⏺ Start Recording'
        status_label.value = 'Stopped. Files saved to recordings/.'
        clip_label.value   = ''


record_button.observe(on_record_toggle, names='value')


def on_new_frame(change):
    """Called every time the camera produces a new frame."""
    if not recording or video_writer is None:
        return

    frame = change['new']                    # BGR8 numpy array
    video_writer.write(frame)

    csv_writer.writerow([
        time.time(),
        current_left,
        current_right,
        current_inputs['throttle'],
        current_inputs['turn'],
        current_inputs['left_trigger'],
        current_inputs['right_trigger'],
    ])


# attach frame callback to camera
camera.observe(on_new_frame, names='value')

RuntimeError: Could not initialize camera.  Please see error trace.

Awesome! Our robot should now respond to our gamepad controller movements.  Now we want to view the live video feed from the camera!

### Create and display Image widget

First, let's display an ``Image`` widget that we'll use to show our live camera feed.  We'll set the ``height`` and ``width``
to just 300 pixels so it doesn't take up too much space.

> FYI: The height and width only effect the rendering on the browser side, not the native image resolution before network transport from robot to browser.

In [ ]:
image = widgets.Image(format='jpeg', width=300, height=300)

display(image)

### Create camera instance

Well, right now there's no image presented, because we haven't set the value yet!  We can do this by creating our ``Camera``
class and attaching the ``value`` attribute of the camera to the ``value attribute of the image.

First, let's create the camera instance, we call the ``instance`` method which will create a new camera
if it hasn't been created yet.  If once already exists, this method will return the existing camera.

In [ ]:
from jetbot import Camera

camera = Camera.instance()

### Connect Camera to Image widget

Our camera class currently only produces values in BGR8 (blue, green, red, 8bit) format, while our image widget accepts values in compressed *JPEG*.
To connect the camera to the image we need to insert the ``bgr8_to_jpeg`` function as a transform in the link.  We do this below

In [ ]:
from jetbot import bgr8_to_jpeg

camera_link = traitlets.dlink((camera, 'value'), (image, 'value'), transform=bgr8_to_jpeg)

You should now see the live video feed shown above!

> REMINDER:  You can right click the output of a cell and select ``Create New View for Output`` to display the cell in a separate window.

### Stop robot if network disconnects

You can drive your robot around by looking through the video feed. But what if your robot disconnects from Wifi?  Well, the motors would keep moving and it would keep trying to stream video and motor commands.  Let's make it so that we stop the robot and unlink the camera and motors when a disconnect occurs.

In [ ]:
from jetbot import Heartbeat


def handle_heartbeat_status(change):
    if change['new'] == Heartbeat.Status.dead:
        camera_link.unlink()
        left_link.unlink()
        right_link.unlink()
        robot.stop()

heartbeat = Heartbeat(period=0.5)

# attach the callback function to heartbeat status
heartbeat.observe(handle_heartbeat_status, names='status')

If the robot disconnects from the internet you'll notice that it stops.  You can then re-connect the camera and motors by re-creating the links with the cell below

In [ ]:
# only call this if your robot links were unlinked, otherwise we'll have redundant links which will double
# the commands transferred

left_link = traitlets.dlink((controller.axes[1], 'value'), (robot.left_motor, 'value'), transform=lambda x: -x)
right_link = traitlets.dlink((controller.axes[3], 'value'), (robot.right_motor, 'value'), transform=lambda x: -x)
camera_link = traitlets.dlink((camera, 'value'), (image, 'value'), transform=bgr8_to_jpeg)

### Save snapshots with gamepad button

Now, we'd like to be able to save some images from our robot.  Let's make it so the right bumper (index 5) saves a snapshot of the current live image.  We'll save the images in the ``snapshots/`` directory, with a name that is guaranteed to be unique using the ``uuid`` python package.  We use the ``uuid1`` identifier, because this also encodes the date and MAC address which we might want to use later.

In [ ]:
import uuid
import subprocess

subprocess.call(['mkdir', '-p', 'snapshots'])

snapshot_image = widgets.Image(format='jpeg', width=300, height=300)

def save_snapshot(change):
    # save snapshot when button is pressed down
    if change['new']:
        file_path = 'snapshots/' + str(uuid.uuid1()) + '.jpg'
        
        # write snapshot to file (we use image value instead of camera because it's already in JPEG format)
        with open(file_path, 'wb') as f:
            f.write(image.value)
            
        # display snapshot that was saved
        snapshot_image.value = image.value


controller.buttons[5].observe(save_snapshot, names='value')

display(widgets.HBox([image, snapshot_image]))
display(controller)

Before closeing this notebook and shutdown the Python kernel for the notebook, we want to properly close the camera connection so that we can use the camera in other notebook.

In [ ]:
camera.stop()

### Conclusion

That's it for this example, have fun!